In [ ]:
CREATE VOLUME IF NOT EXISTS workspace.default.lantern_walks;

In [ ]:
%python
from pathlib import Path

WALK_ROOT = Path("/Volumes/workspace/default/lantern_walks")

for name in ["incoming", "rgb", "depth", "masks", "exports"]:
    (WALK_ROOT / name).mkdir(parents=True, exist_ok=True)

print("Lantern walk storage ready")

for folder in WALK_ROOT.iterdir():
    print(folder)

In [ ]:
%python
"""Normalize Lantern exports without Spark or fabricated sensor/model evidence.

This module is embedded verbatim in lantern_lakehouse.ipynb so Databricks workers
do not need a separate local Python module installed.
"""
import base64
import json
import math

MAX_PIXELS = 512 * 512


def number(value):
    return value if isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value) else None


def probability(value):
    value = number(value)
    return value if value is not None and 0 <= value <= 1 else None


def pose_matrix(value):
    if not isinstance(value, list) or len(value) != 16 or any(number(n) is None for n in value):
        return None
    # A rigid ARKit transform, stored column-major. Reject malformed scale/basis.
    axes = [value[start:start + 3] for start in (0, 4, 8)]
    if any(abs(sum(n * n for n in axis) - 1) > 0.02 for axis in axes):
        return None
    if any(abs(sum(axes[a][i] * axes[b][i] for i in range(3))) > 0.02 for a, b in ((0, 1), (0, 2), (1, 2))):
        return None
    if any(abs(value[i]) > 1e-5 for i in (3, 7, 11)) or abs(value[15] - 1) > 1e-5:
        return None
    return value


def world_point(point, pose):
    if not pose or not isinstance(point, list) or len(point) != 3 or any(number(n) is None for n in point):
        return None
    hx, hz = -pose[8], -pose[10]
    length = math.hypot(hx, hz)
    if length <= 0.4:
        return None
    hx, hz = hx / length, hz / length
    right, up, forward = point
    return [pose[12] - hz * right + hx * forward, pose[13] + up,
            pose[14] + hx * right + hz * forward]


def normalize_record(record):
    if not isinstance(record, dict):
        raise ValueError('record must be an object')
    if record.get('consent', {}).get('granted') is False:
        raise ValueError('recording consent was not granted')
    # Existing flat JSONL recordings remain supported without relabelling data.
    if record.get('source_type') == 'iphone_recording':
        result = dict(record)
        people = result.get('people') or []
        result['people'] = [dict(person, coordinate_space_id=person.get('coordinate_space_id', 'legacy'))
                            for person in people if isinstance(person, dict)]
        return result
    if record.get('schemaVersion') != 1 or record.get('consent', {}).get('granted') is not True:
        raise ValueError('expected a consented telemetry v1 export')
    frame = record.get('frame') or {}
    observation = frame.get('observation') or {}
    timestamp = number(observation.get('timeMs'))
    sequence = record.get('sequence')
    session = record.get('sessionId')
    if observation.get('source') != 'arkit' or not isinstance(session, str) or not session:
        raise ValueError('not a real iPhone recording')
    if timestamp is None or number(sequence) is None or sequence < 0 or int(sequence) != sequence:
        raise ValueError('missing or invalid frame identity/timestamp')
    result = dict(source_type='iphone_recording', session_id=session, frame_id=int(sequence),
                  captured_at_ms=int(timestamp), camera_pose_matrix=pose_matrix(observation.get('pose')),
                  rgb_path=None, depth_path=None, rgb_ts_ms=None, depth_ts_ms=None,
                  glass_probability=None, stair_riser_probability=None, stair_tread_probability=None,
                  people=[], masks=[], adapter_warnings=[])
    warnings = result['adapter_warnings']
    depth = observation.get('depth')
    if isinstance(depth, dict):
        width, height = depth.get('width'), depth.get('height')
        if (type(width) is int and type(height) is int and width > 0 and height > 0
                and width * height <= MAX_PIXELS and isinstance(depth.get('meters'), list)
                and len(depth['meters']) == width * height):
            meters = [n if number(n) is not None and n > 0 else None for n in depth['meters']]
            confidence = depth.get('confidence')
            if not isinstance(confidence, list) or len(confidence) != len(meters):
                confidence = [None] * len(meters)
            intrinsics = depth.get('intrinsics')
            if (not isinstance(intrinsics, list) or len(intrinsics) != 4
                    or any(number(n) is None for n in intrinsics) or min(intrinsics[:2]) <= 0):
                intrinsics = None
                warnings.append('invalid_depth_intrinsics')
            result['inline_depth'] = dict(width=width, height=height, meters=meters,
                                          confidence=[n if type(n) is int and n in (0, 1, 2) else None for n in confidence],
                                          intrinsics=intrinsics)
            result['depth_ts_ms'] = int(timestamp)
            result['depth_valid_fraction'] = sum(n is not None for n in meters) / len(meters)
        else:
            warnings.append('invalid_embedded_depth')
    warnings.append('rgb_not_exported')
    preview = frame.get('modelPreview')
    if not isinstance(preview, dict) or number(preview.get('timeMs')) is None:
        warnings.append('model_preview_not_recorded')
        return result
    prediction_time = preview['timeMs']
    result['prediction_ts_ms'] = int(prediction_time)
    result['prediction_sensor_gap_ms'] = abs(prediction_time - timestamp)
    versions = preview.get('modelVersions') or {}
    for kind, key in [('glass', 'glass'), ('stairs', 'stair'), ('person', 'people')]:
        version = versions.get(kind)
        result[key + '_model_version'] = version if isinstance(version, str) and version else None
    if any(result[key + '_model_version'] is None for key in ('glass', 'stair', 'people')):
        warnings.append('model_versions_unavailable')
    # A preview has its own camera pose. Older exports can use the observation
    # pose only if they refer to the SAME captured frame, not just a recent one.
    preview_pose = pose_matrix(preview.get('cameraPose'))
    if preview_pose is None and abs(prediction_time - timestamp) <= 1:
        preview_pose = result['camera_pose_matrix']
    if preview_pose is None:
        warnings.append('prediction_pose_unavailable')
    result['prediction_camera_pose_matrix'] = preview_pose
    for mask in preview.get('masks') or []:
        if not isinstance(mask, dict) or mask.get('kind') not in ('glass', 'stairs'):
            continue
        size, labels = mask.get('size'), mask.get('labels')
        if not isinstance(size, int) or size not in (96, 224, 384) or not isinstance(labels, str):
            warnings.append('invalid_mask')
            continue
        try:
            if len(labels) > 4 * math.ceil(MAX_PIXELS / 3):
                raise ValueError('mask too large')
            decoded = base64.b64decode(labels, validate=True)
            allowed = (0, 1) if mask['kind'] == 'glass' else (0, 1, 2)
            if len(decoded) != size * size or any(label not in allowed for label in decoded):
                raise ValueError('invalid mask raster')
        except (ValueError, TypeError):
            warnings.append('invalid_mask')
            continue
        result['masks'].append(dict(kind=mask['kind'], size=size, labels=labels))
        if mask['kind'] == 'glass':
            result['glass_mask_fraction'] = decoded.count(1) / len(decoded)
        else:
            result['stair_riser_mask_fraction'] = decoded.count(1) / len(decoded)
            result['stair_tread_mask_fraction'] = decoded.count(2) / len(decoded)
    if result['masks']:
        warnings.append('mask_probabilities_unavailable')
    for person in (preview.get('people') or [])[:16]:
        if not isinstance(person, dict):
            continue
        box = person.get('box')
        if not isinstance(box, list) or len(box) != 4 or any(probability(n) is None for n in box):
            continue
        world = world_point(person.get('position'), preview_pose)
        local = person.get('position')
        located = world or (local if isinstance(local, list) and len(local) == 3
                           and all(number(n) is not None for n in local) else None)
        track_id = person.get('trackId')
        item = dict(track_id=str(track_id) if isinstance(track_id, int) and not isinstance(track_id, bool) and track_id > 0 else None,
                    confidence=probability(person.get('confidence')), bbox_xywh=box,
                    coordinate_frame='arkit_world' if world else 'phone_relative' if located else 'image',
                    coordinate_space_id=str(preview['trackingSession']) if preview.get('trackingSession') is not None else None,
                    x_m=located[0] if located else None, z_m=located[2] if located else None,
                    vx_mps=None, vz_mps=None)
        horizon = number(person.get('predictionHorizonMs'))
        future = world_point(person.get('predictedPosition'), preview_pose)
        if world and future and horizon is not None and 0 < horizon <= 1000:
            item['vx_mps'] = (future[0] - world[0]) * 1000 / horizon
            item['vz_mps'] = (future[2] - world[2]) * 1000 / horizon
        result['people'].append(item)
    return result


def normalize_json(raw):
    """Spark UDF boundary. Bronze retains rejected source lines for inspection."""
    try:
        return json.dumps(normalize_record(json.loads(raw)), allow_nan=False, separators=(',', ':'))
    except (ValueError, TypeError, KeyError, AttributeError, OverflowError):
        return None


In [ ]:

%python
from pathlib import Path
from pyspark.sql import functions as F, Window

ROOT = Path("/Volumes/workspace/default/lantern_walks")
INCOMING = ROOT / "incoming"
DB = "workspace.default"

BRONZE = f"{DB}.lantern_bronze"
SILVER = f"{DB}.lantern_silver"
GOLD = f"{DB}.lantern_gold_hazards"
REVIEW = f"{DB}.lantern_review_candidates"
TRACKS = f"{DB}.lantern_pedestrian_tracks"
SESSIONS = f"{DB}.lantern_sessions"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE} (
    record_hash STRING,
    raw_json STRING,
    source_file STRING,
    ingested_at TIMESTAMP
) USING DELTA
""")

incoming_files = sorted(
    str(p)
    for p in INCOMING.rglob("*")
    if p.suffix.lower() in (".jsonl", ".ndjson")
    if p.is_file()
    and not p.name.upper().startswith("SIM_")
)

if incoming_files:
    new = (
        spark.read.text(incoming_files)
        .select(
            F.col("value").alias("raw_json"),
            F.col("_metadata.file_path").alias("source_file")
        )
        .where(F.length(F.trim("raw_json")) > 0)
        .withColumn("record_hash", F.sha2("raw_json", 256))
        .withColumn("ingested_at", F.current_timestamp())
        .dropDuplicates(["record_hash"])
    )

    new.createOrReplaceTempView("lantern_new_records")

    spark.sql(f"""
        MERGE INTO {BRONZE} t
        USING lantern_new_records s
        ON t.record_hash = s.record_hash
        WHEN NOT MATCHED THEN INSERT (
            record_hash,
            raw_json,
            source_file,
            ingested_at
        )
        VALUES (
            s.record_hash,
            s.raw_json,
            s.source_file,
            s.ingested_at
        )
    """)

frame_schema = """
source_type STRING,
session_id STRING,
frame_id BIGINT,
captured_at_ms BIGINT,
rgb_path STRING,
depth_path STRING,
rgb_ts_ms BIGINT,
depth_ts_ms BIGINT,
depth_valid_fraction DOUBLE,
camera_pose STRUCT<
    tx_m:DOUBLE,
    ty_m:DOUBLE,
    tz_m:DOUBLE,
    qx:DOUBLE,
    qy:DOUBLE,
    qz:DOUBLE,
    qw:DOUBLE
>,
glass_model_version STRING,
stair_model_version STRING,
people_model_version STRING,
glass_probability DOUBLE,
stair_riser_probability DOUBLE,
stair_tread_probability DOUBLE,
glass_mask_path STRING,
stair_mask_path STRING,
prediction_ts_ms BIGINT,
prediction_sensor_gap_ms DOUBLE,
camera_pose_matrix ARRAY<DOUBLE>,
prediction_camera_pose_matrix ARRAY<DOUBLE>,
inline_depth STRUCT<width:INT,height:INT,meters:ARRAY<DOUBLE>,confidence:ARRAY<INT>,intrinsics:ARRAY<DOUBLE>>,
masks ARRAY<STRUCT<kind:STRING,size:INT,labels:STRING>>,
glass_mask_fraction DOUBLE,
stair_riser_mask_fraction DOUBLE,
stair_tread_mask_fraction DOUBLE,
adapter_warnings ARRAY<STRING>,
people ARRAY<STRUCT<
    track_id:STRING,
    confidence:DOUBLE,
    bbox_xywh:ARRAY<DOUBLE>,
    coordinate_frame:STRING,
    coordinate_space_id:STRING,
    x_m:DOUBLE,
    z_m:DOUBLE,
    vx_mps:DOUBLE,
    vz_mps:DOUBLE
>>
"""

w = Window.partitionBy(
    "session_id", "frame_id"
).orderBy(
    F.desc("ingested_at"),
    F.desc("record_hash")
)

normalize_record_udf = F.udf(normalize_json, "string")
normalized = spark.table(BRONZE).withColumn("normalized_json", normalize_record_udf("raw_json"))
rejected_count = normalized.where(F.col("normalized_json").isNull()).count()
print("Rejected records retained in Bronze:", rejected_count)

silver = (
    normalized
    .withColumn("frame", F.from_json("normalized_json", frame_schema))
    .select(
        "record_hash",
        "source_file",
        "ingested_at",
        "frame.*"
    )
    .where(
        (F.col("source_type") == "iphone_recording")
        & F.col("session_id").isNotNull()
        & F.col("frame_id").isNotNull()
        & F.col("captured_at_ms").isNotNull()
    )
    .withColumn("_rank", F.row_number().over(w))
    .where(F.col("_rank") == 1)
    .drop("_rank")
    .withColumn(
        "rgb_depth_gap_ms",
        F.abs(F.col("rgb_ts_ms") - F.col("depth_ts_ms"))
    )
    .withColumn(
        "timestamps_close",
        F.coalesce(
            F.col("rgb_depth_gap_ms") <= 80,
            F.lit(False)
        )
    )
    .withColumn(
        "people_count",
        F.coalesce(F.size("people"), F.lit(0))
    )
)

asset_paths = [
    r["path"]
    for r in (
        silver.select(F.col("rgb_path").alias("path"))
        .union(
            silver.select(F.col("depth_path").alias("path"))
        )
    ).where(F.col("path").isNotNull()).distinct().collect()
]

if len(asset_paths) > 50000:
    raise RuntimeError("Too many asset paths in one batch")

flags = spark.createDataFrame(
    [
        (
            p,
            Path(p).resolve().is_relative_to(ROOT.resolve())
            and Path(p).is_file()
        )
        for p in asset_paths
    ],
    "path STRING, exists BOOLEAN"
)

rgb_flags = flags.select(
    F.col("path").alias("rgb_path"),
    F.col("exists").alias("rgb_exists")
)

depth_flags = flags.select(
    F.col("path").alias("depth_path"),
    F.col("exists").alias("depth_exists")
)

silver = (
    silver
    .join(F.broadcast(rgb_flags), "rgb_path", "left")
    .join(F.broadcast(depth_flags), "depth_path", "left")
    .fillna({
        "rgb_exists": False,
        "depth_exists": False
    })
    .withColumn("embedded_depth_available", F.col("inline_depth").isNotNull())
    .withColumn("depth_exists", F.col("depth_exists") | F.col("embedded_depth_available"))
    .withColumn(
        "rgb_depth_aligned",
        F.col("rgb_exists")
        & F.col("depth_exists")
        & F.col("timestamps_close")
    )
    .withColumn(
        "data_quality",
        F.when(
            F.col("rgb_depth_aligned"),
            "metadata_pair_valid"
        )
        .when(
            F.col("embedded_depth_available") & ~F.col("rgb_exists"),
            "embedded_depth_no_rgb"
        )
        .when(
            ~F.col("rgb_exists") | ~F.col("depth_exists"),
            "missing_file"
        )
        .otherwise("timestamp_mismatch")
    )
)

def save(df, table):
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )

save(silver, SILVER)
silver = spark.table(SILVER)

stair_score = F.greatest(
    "stair_riser_probability",
    "stair_tread_probability"
)

def hazard_rows(kind, condition, score, version, mask_fraction=None):
    return silver.where(condition).select(
        "session_id",
        "frame_id",
        F.coalesce("prediction_ts_ms", "captured_at_ms").alias("captured_at_ms"),
        "rgb_path",
        "depth_path",
        "rgb_depth_aligned",
        "source_file",
        F.lit(kind).alias("hazard_type"),
        score.cast("double").alias("confidence"),
        (mask_fraction if mask_fraction is not None else F.lit(None)).cast("double").alias("mask_fraction"),
        F.when(score.isNotNull(), F.lit("model_probability"))
         .otherwise(F.lit("person_detection" if kind == "person" else "mask_labels")).alias("evidence_type"),
        F.col(version).alias("model_version")
    )

glass_events = hazard_rows(
    "glass",
    (F.col("glass_probability") >= 0.65) | (F.col("glass_mask_fraction") > 0),
    F.col("glass_probability"),
    "glass_model_version",
    F.col("glass_mask_fraction")
)

stair_events = hazard_rows(
    "stairs",
    (stair_score >= 0.65) | (F.greatest("stair_riser_mask_fraction", "stair_tread_mask_fraction") > 0),
    stair_score,
    "stair_model_version",
    F.col("stair_riser_mask_fraction") + F.col("stair_tread_mask_fraction")
)

person_events = hazard_rows(
    "person",
    F.col("people_count") > 0,
    F.expr("array_max(transform(people, p -> p.confidence))"),
    "people_model_version"
)

gold = (
    glass_events
    .unionByName(stair_events)
    .unionByName(person_events)
    .dropDuplicates(["session_id", "captured_at_ms", "hazard_type"])
)

save(gold, GOLD)

reasons = F.array(
    F.when(
        (F.col("glass_mask_fraction") > 0) & (F.col("depth_valid_fraction") < 0.5),
        F.lit("glass_mask_low_depth_validity")
    ),
    F.when(
        ~F.col("rgb_depth_aligned"),
        F.lit("missing_or_misaligned_sensor_files")
    ),
    F.when(
        F.col("glass_probability").between(0.35, 0.65),
        F.lit("uncertain_glass")
    ),
    F.when(
        stair_score.between(0.35, 0.65),
        F.lit("uncertain_stairs")
    ),
    F.when(
        (F.col("glass_probability") >= 0.65)
        & (F.col("depth_valid_fraction") < 0.5),
        F.lit("glass_low_depth_validity")
    ),
    *[
        F.when(F.array_contains("adapter_warnings", reason), F.lit(reason))
        for reason in ["model_preview_not_recorded", "model_versions_unavailable",
                       "mask_probabilities_unavailable", "prediction_pose_unavailable", "invalid_embedded_depth", "invalid_depth_intrinsics", "invalid_mask"]
    ]
)

reviews = (
    silver
    .withColumn("reason", F.explode(reasons))
    .where(F.col("reason").isNotNull())
    .select(
        "session_id",
        "frame_id",
        "captured_at_ms",
        "rgb_path",
        "depth_path",
        "reason",
        "glass_probability",
        "depth_valid_fraction",
        "source_file"
    )
)

save(reviews, REVIEW)

track_w = Window.partitionBy(
    "session_id",
    "track_id",
    "coordinate_space_id"
).orderBy(
    "captured_at_ms",
    "frame_id"
)

tracks = (
    silver
    .select(
        "session_id",
        "frame_id",
        F.coalesce("prediction_ts_ms", "captured_at_ms").alias("captured_at_ms"),
        "source_file",
        F.explode_outer("people").alias("person")
    )
    .where(F.col("person.track_id").isNotNull())
    .select(
        "session_id",
        "frame_id",
        "captured_at_ms",
        "source_file",
        F.col("person.track_id").alias("track_id"),
        F.col("person.confidence").alias("confidence"),
        F.col("person.bbox_xywh").alias("bbox_xywh"),
        F.col("person.coordinate_frame").alias("coordinate_frame"),
        F.col("person.coordinate_space_id").alias("coordinate_space_id"),
        F.col("person.x_m").alias("x_m"),
        F.col("person.z_m").alias("z_m"),
        F.col("person.vx_mps").alias("supplied_vx"),
        F.col("person.vz_mps").alias("supplied_vz")
    )
    .dropDuplicates(["session_id", "track_id", "coordinate_space_id", "captured_at_ms"])
    .withColumn(
        "dt",
        (
            F.col("captured_at_ms")
            - F.lag("captured_at_ms").over(track_w)
        ) / 1000
    )
    .withColumn("prev_x", F.lag("x_m").over(track_w))
    .withColumn("prev_z", F.lag("z_m").over(track_w))
    .withColumn(
        "prev_frame",
        F.lag("coordinate_frame").over(track_w)
    )
    .withColumn(
        "world_ok",
        (F.col("coordinate_frame") == "arkit_world")
        & F.col("x_m").isNotNull()
        & F.col("z_m").isNotNull()
    )
    .withColumn(
        "motion_ok",
        F.col("world_ok")
        & F.col("coordinate_space_id").isNotNull()
        & (F.col("prev_frame") == "arkit_world")
        & F.col("dt").between(0.001, 2.0)
    )
    .withColumn(
        "vx_mps",
        F.when(
            F.col("world_ok"),
            F.coalesce(
                "supplied_vx",
                F.when(
                    F.col("motion_ok"),
                    (
                        F.col("x_m") - F.col("prev_x")
                    ) / F.col("dt")
                )
            )
        )
    )
    .withColumn(
        "vz_mps",
        F.when(
            F.col("world_ok"),
            F.coalesce(
                "supplied_vz",
                F.when(
                    F.col("motion_ok"),
                    (
                        F.col("z_m") - F.col("prev_z")
                    ) / F.col("dt")
                )
            )
        )
    )
    .withColumn(
        "predicted_x_m_1s",
        F.when(
            F.col("world_ok"),
            F.col("x_m") + F.col("vx_mps")
        )
    )
    .withColumn(
        "predicted_z_m_1s",
        F.when(
            F.col("world_ok"),
            F.col("z_m") + F.col("vz_mps")
        )
    )
    .select(
        "session_id",
        "frame_id",
        "captured_at_ms",
        "track_id",
        "confidence",
        "bbox_xywh",
        "coordinate_frame",
        "coordinate_space_id",
        "x_m",
        "z_m",
        "vx_mps",
        "vz_mps",
        "predicted_x_m_1s",
        "predicted_z_m_1s",
        "source_file"
    )
)

save(tracks, TRACKS)

sessions = (
    silver.groupBy("session_id")
    .agg(
        F.count("*").alias("frame_count"),
        F.sum(
            F.col("rgb_depth_aligned").cast("int")
        ).alias("metadata_aligned_frames"),
        F.avg(
            "depth_valid_fraction"
        ).alias("average_depth_valid_fraction")
    )
    .join(
        spark.table(GOLD)
        .groupBy("session_id")
        .agg(F.count("*").alias("hazard_events")),
        "session_id",
        "left"
    )
    .join(
        spark.table(REVIEW)
        .groupBy("session_id")
        .agg(F.count("*").alias("review_candidates")),
        "session_id",
        "left"
    )
    .fillna({
        "hazard_events": 0,
        "review_candidates": 0
    })
)

save(sessions, SESSIONS)

print("LANTERN LAKEHOUSE READY")

for table in [
    BRONZE,
    SILVER,
    GOLD,
    REVIEW,
    TRACKS,
    SESSIONS
]:
    print(
        table,
        spark.table(table).count(),
        "rows"
    )

if not incoming_files:
    print(
        "Waiting for real iPhone recordings in:",
        INCOMING
    )
else:
    print(
        "Real recording files found:",
        len(incoming_files)
    )

print("iPhone embedded-depth exports do not contain full RGB training images. Mask fractions are label coverage, not probabilities.")
